# 第三部分：RAII 与资源生命周期

## 实验 6：封装完整的 `File` 资源类

前五个实验依次发现手动释放问题、建立最小 RAII wrapper，并验证作用域、提前返回和异常路径。本实验把这些规则汇总为一个接口与实现分离的小型 `File` 类型。

这里的“完整”是指当前 RAII 阶段所需的最小闭环：构造时建立有效资源、析构时兜底释放、禁止错误复制、封装常用操作、传播显式操作错误，并提供受约束的底层 handle 借用。移动语义将在后续阶段补充。

### 1. 文件结构

```text
03-raii/
├── 06_file_class.ipynb    # 实现、使用与设计说明
└── include/
    └── file.hpp           # File 的公开接口
```

打开 [file.hpp](include/file.hpp) 对照后续代码。Notebook 中的成员函数定义相当于真实工程里的 `file.cpp`。

### 2. 公开接口表达所有权策略

头文件不只列出函数，也应让类型的资源语义清晰可见：

- 构造成功意味着拥有一个有效 `std::FILE*`；
- 析构函数 `noexcept`，负责最终关闭；
- 复制与移动暂时都被禁止，确保唯一所有权；
- `write()` 和 `flush()` 封装常用操作并报告失败；
- `get()` 只提供借用指针，用于尚未包装的 C API。

先载入头文件中的类定义：

In [ ]:
#include "./include/file.hpp"

#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <type_traits>

### 3. 实现资源获取与释放

构造函数先校验参数，再获取文件。如果打开失败就抛出异常，因此不存在“已经构造完成但内部句柄无效”的 `File`。析构函数只做不抛异常的兜底关闭。

In [ ]:
File::File(
    const char *path,
    const char *mode)
    : file_(nullptr),
      path_(path == nullptr ? "" : path)
{
    if (path == nullptr || mode == nullptr)
    {
        throw std::invalid_argument("path and mode must not be null");
    }

    file_ = std::fopen(path, mode);

    if (file_ == nullptr)
    {
        throw std::runtime_error("cannot open file: " + path_);
    }
}

File::~File() noexcept
{
    if (file_ != nullptr)
    {
        std::fclose(file_);
    }
}

void File::write(const char *text)
{
    if (text == nullptr)
    {
        throw std::invalid_argument("text must not be null");
    }

    if (std::fputs(text, file_) == EOF)
    {
        throw std::runtime_error("cannot write file: " + path_);
    }
}

void File::flush()
{
    if (std::fflush(file_) != 0)
    {
        throw std::runtime_error("cannot flush file: " + path_);
    }
}

std::FILE *File::get() const noexcept
{
    return file_;
}

const std::string &File::path() const noexcept
{
    return path_;
}

### 4. 使用高层接口写入文件

调用方只表达打开、写入和刷新，不直接管理关闭动作。所有输出统一放在 `outputs/06/`。

In [ ]:
std::filesystem::create_directories("outputs/06");

void write_message()
{
    File file(
        "outputs/06/message.txt",
        "w");

    file.write("Resource lifetime follows object lifetime.");
    file.flush();

    std::cout << "written: " << file.path() << std::endl;
}

write_message();

`flush()` 是显式操作，因此可以通过异常报告失败；析构函数仍只负责不抛异常的最终关闭。这体现了资源类常见的双层设计：

```text
显式操作：允许报告业务可处理的错误
析构清理：noexcept 的最后防线
```

### 5. 为 C API 提供受约束的借用

有些 C API 尚未被 `File` 封装，可以通过 `get()` 临时借用底层 handle。借用者不能关闭它，也不能让指针活得比 `File` 更久。

In [ ]:
{
    File file(
        "outputs/06/c-api.txt",
        "w");

    std::fputs("written through borrowed FILE pointer", file.get());
}

`get()` 是兼容性逃生口，也会暂时削弱封装。若调用方对返回值执行 `std::fclose()`，`File` 的不变量就会被破坏。生产类型应尽量把常用操作封装为成员函数，只在确有 C 互操作需求时暴露借用 handle。

### 6. 用类型特征检查所有权策略

当前 `File` 是唯一且不可转移的 owner。复制会造成 double close；移动将在后续阶段实现，因此这里显式禁止两者。

In [ ]:
std::cout
    << std::boolalpha
    << "copy constructible: "
    << std::is_copy_constructible_v<File>
    << std::endl
    << "move constructible: "
    << std::is_move_constructible_v<File>
    << std::endl;

### 7. 接口与实现分离的意义

调用者包含 `file.hpp` 后，可以看见类型大小、公开操作和所有权限制，却不需要阅读每个函数如何调用 C API。真实工程会把本 Notebook 的成员函数定义放入 `file.cpp`：

```text
include/file.hpp → 编译期契约
file.cpp         → 资源策略实现
使用方 .cpp      → 只依赖公开接口
```

资源类因此成为 C API 与上层业务代码之间的小型边界。

### 8. RAII 与 Garbage Collection 不是一回事

GC 主要决定不可达内存何时回收；文件、Socket、锁和 GPU buffer 等资源不能简单等待未来某次 GC。RAII 提供的是确定性资源管理：

```text
GC
object unreachable → sometime later → memory reclaimed

RAII
scope ends → immediately run destructor → resource released
```

Kotlin 的 `file.use { ... }` 也在表达确定的使用范围；C++ 则把这一模型统一放进对象生命周期与析构机制中。

### 9. RAII 类型可以组合成 SDK 对象

真实 Native SDK 的高层对象通常拥有多个更小的 RAII 成员：

```cpp
class Runtime
{
public:
    Runtime();
    ~Runtime() = default;

private:
    DeviceHandle device_;
    ModelHandle model_;
    Buffer input_;
    Buffer output_;
};
```

成员依次构造、逆序析构；构造中途失败时，已构造成员自动回滚。`Runtime` 不需要手写一长串清理分支，这就是 RAII 在 SDK 内部的真正价值。

### 10. 跨语言边界上的三层资源模型

C ABI 不能直接表达 C++ 析构语义，因此对外通常重新暴露显式 create/destroy：

```text
C++ 内部       RAII owner
                   ↓
C ABI          create / destroy
                   ↓
Kotlin wrapper AutoCloseable / use
```

```c
sdk_runtime_t *sdk_runtime_create(void);
void sdk_runtime_destroy(sdk_runtime_t *runtime);
```

C++ 实现内部仍可完全使用 RAII；C ABI 转换为显式生命周期协议；Kotlin wrapper 再用 `AutoCloseable` 和 `use` 恢复结构化资源管理。

### 11. 当前 `File` 的设计边界

这个类型刻意保持小而清晰，仍未处理：

- 移动构造和移动赋值；
- 二进制读写、定位和完整读取接口；
- 错误码中保留 `errno` 等系统诊断；
- 线程安全；
- 原子替换文件等强异常保证。

这些不是 RAII 基本模型的缺陷，而是更高层 API 契约。先确保唯一所有权与确定性清理，再按真实需求扩展。

### 本阶段结论

RAII 的本质是确定性资源管理：类型表达所有权，构造函数建立有效状态，析构函数结束资源生命周期，作用域与栈展开保证所有正常返回和异常路径都执行清理。

以后遇到 `FILE*`、Socket、Native handle、锁或 GPU buffer，应先问：谁拥有它？所有权何时开始和结束？能否把这段生命周期绑定到一个不可错误复制的 C++ 对象？